# Calculate Difensible Space

Purpose:
Processes building defensible space polygon data by clipping associated land cover raster tiles to each polygon and counting land cover classes.

Key Steps:

    Setup:
        Imports libraries for geospatial operations, raster processing, plotting, and parallel processing.
        Configures file paths, processing parameters (e.g., buffer sizes, color mappings), and logging.

    Processing Functions:
        process_polygon: Clips a raster to a polygon and counts unique land cover values.
        plot_raster_and_polygons: Optionally visualizes the raster with overlaid polygons for quality control.
        process_polygon_file: Reads a feather file of polygons, reprojects geometries to match the raster CRS, splits the data into chunks, and processes each chunk in parallel (using process_part).
        process_part: Processes a chunk of polygons to aggregate land cover counts.

    Workflow:
        Iterates over polygon files for each buffer type ("10m" and "30m").
        Optionally plots a subset for quality control.
        Processes each file by clipping the raster, counting classifications, and saving the results as CSV.

In [ ]:
import os
WRI_PROJECT_ROOT = os.environ.get("WRI_PROJECT_ROOT", "/home/shares/wwri-wildfire")

import warnings
import logging
import os
import random
import geopandas as gpd
import pyarrow.feather as feather
import rasterio
import rasterio.mask
import pandas as pd
import numpy as np
from shapely import wkb
from tqdm import tqdm
from multiprocessing import Pool
import matplotlib.pyplot as plt
from pyproj import Transformer
from shapely.ops import transform
import matplotlib.colors as mcolors

# Suppress specific FutureWarning
warnings.simplefilter(action='ignore', category=FutureWarning)

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# -----------------------------
# Configuration: File Paths
# -----------------------------

# Base directory for built environment domain data
BASE_DIR = os.path.join(WRI_PROJECT_ROOT, 'data', 'built-environment-domain-data', 'defensible-space')

# Polygon directories for different buffer types
POLYGON_DIRS = {
    "10m": os.path.join(BASE_DIR, "building_defensible_space_polygons/zoom_7_buffer_10m"),
    "30m": os.path.join(BASE_DIR, "building_defensible_space_polygons/zoom_7_buffer_30m")
}

# Directory containing raster land cover tiles (Updated Path)
TILE_DIR = os.path.join(WRI_PROJECT_ROOT, 'data', 'multi-domain-data', 'esri-land-use-land-cover-10m')

# Output directories for CSVs based on buffer type
OUTPUT_CSV_DIRS = {
    "10m": os.path.join(BASE_DIR, "structure-landcover-counts/10m"),
    "30m": os.path.join(BASE_DIR, "structure-landcover-counts/30m")
}

# Ensure output directories exist
for dir_path in OUTPUT_CSV_DIRS.values():
    os.makedirs(dir_path, exist_ok=True)

# -----------------------------
# Configuration: Processing Parameters
# -----------------------------

# Land cover classification names
CLASSIFICATION_NAMES = {
    0: 'No Data 1',
    1: 'Water',
    2: 'Trees',
    4: 'Flooded vegetation',
    5: 'Crops',
    7: 'Built area',
    8: 'Bare ground',
    9: 'Snow/ice',
    10: 'Clouds',
    11: 'Rangeland',
    255: 'No Data 2',
    100: 'No Data 3'
}

# Color mapping for land cover classifications
COLOR_KEY = {
    0: '#FFA500',   # No Data 1 (bright orange)
    1: '#0000FF',   # Water
    2: '#00FF00',   # Trees
    4: '#800080',   # Flooded vegetation
    5: '#FFFF00',   # Crops
    7: '#808080',   # Built area
    8: '#8B4513',   # Bare ground
    9: '#FFFFFF',   # Snow/ice
    10: '#D3D3D3',  # Clouds
    11: '#D2B48C',  # Rangeland
    255: '#FF00FF', # No Data 2 (bright purple)
    100: '#000000'  # No Data 3 (black)
}

# Create a colormap based on the color key
CMAP = mcolors.ListedColormap([COLOR_KEY[key] for key in sorted(COLOR_KEY.keys())])

# Multiprocessing configuration
NUM_WORKERS = 65
SPLIT_SIZE = 500

# Quality control plotting
PLOT_PERCENT = 0  # Set desired percentage (e.g., 10 for 10%)

# Overwrite existing CSVs
OVERWRITE = True

# -----------------------------
# Function Definitions
# -----------------------------

def process_polygon(polygon, unique_id, raster_path):
    try:
        with rasterio.open(raster_path) as src:
            # Define the new no-data value
            new_nodata_value = 100
            
            # Clip the raster to the polygon extent and set the new no-data value
            out_image, out_transform = rasterio.mask.mask(
                src, [polygon], crop=True, nodata=new_nodata_value, all_touched=True
            )
            
            # Explicitly set the values outside the polygon to the new no-data value
            out_image[out_image == src.nodata] = new_nodata_value
            
            # Flatten the array to count unique values, including the no-data value
            flat_data = out_image[0].flatten()
            unique, counts = np.unique(flat_data, return_counts=True)
            counts_dict = dict(zip(unique, counts))
            
            # Print for debugging
            logging.debug(f"Unique counts for polygon {unique_id}: {counts_dict}")
        
        return unique_id, counts_dict
    except Exception as e:
        logging.error(f"Error processing polygon {unique_id}: {e}")
        return None

def plot_raster_and_polygons(raster_path, gdf):
    try:
        logging.info(f"Opening raster file: {raster_path}")
        with rasterio.open(raster_path) as src:
            raster_image = src.read(1)
            raster_transform = src.transform
            raster_crs = src.crs
            print(f"Raster CRS: {raster_crs}")  # Print statement for raster CRS
            
            # Check and reproject GeoDataFrame CRS if necessary
            if gdf.crs != raster_crs:
                logging.info(f"GeoDataFrame CRS ({gdf.crs}) does not match raster CRS ({raster_crs}). Reprojecting GeoDataFrame...")
                gdf = gdf.to_crs(raster_crs)

            plt.figure(figsize=(10, 10))
            extent = (
                raster_transform[2],
                raster_transform[2] + raster_transform[0] * src.width,
                raster_transform[5] + raster_transform[4] * src.height,
                raster_transform[5]
            )

            if not all(map(np.isfinite, extent)):
                logging.warning("Extent contains non-finite values, skipping plot.")
                return

            # Manually map colors to categories
            unique_vals = np.unique(raster_image)
            mapped_colors = np.zeros_like(raster_image, dtype=np.uint8)
            for val in unique_vals:
                if val in COLOR_KEY:
                    mapped_colors[raster_image == val] = list(COLOR_KEY.keys()).index(val)

            plt.imshow(mapped_colors, cmap=CMAP, extent=extent)

            if not gdf.is_valid.all():
                logging.warning("Invalid geometries found in GeoDataFrame, attempting to fix...")
                gdf = gdf.buffer(0)

            gdf.plot(ax=plt.gca(), facecolor='none', edgecolor='red')

            # Create custom legend
            handles = [plt.Line2D([0], [0], color=COLOR_KEY[key], lw=4) for key in COLOR_KEY]
            labels = [CLASSIFICATION_NAMES[key] for key in COLOR_KEY]
            plt.legend(handles, labels, loc='upper right', title='Land Cover')

            plt.show()
            plt.close()

            # Count raster classification types
            unique, counts = np.unique(raster_image, return_counts=True)
            classification_counts = dict(zip(unique, counts))
            logging.info(f"Raster classification counts for {raster_path}:")
            for classification, count in classification_counts.items():
                if classification in CLASSIFICATION_NAMES:
                    logging.info(f"{CLASSIFICATION_NAMES[classification]}: {count}")
            
            # Debug step: List all unique cell values in the raster
            unique_values_in_raster = np.unique(raster_image)
            logging.debug(f"Unique cell values in the raster for {raster_path}: {unique_values_in_raster}")

    except Exception as e:
        logging.error(f"Error plotting raster and polygons: {e}")

def process_row(row, raster_path, geometry_column):
    try:
        return process_polygon(row[geometry_column], row['unique_id'], raster_path)
    except Exception as e:
        logging.error(f"Error processing polygon: {e}")
        return None

def process_polygon_file(feather_file, file_index, total_files, buffer_type):
    logging.info(f"Processing polygon file {file_index + 1} of {total_files}: {feather_file}")

    # Check if the CSV file already exists
    quadkey = os.path.splitext(feather_file)[0]
    output_csv_path = os.path.join(OUTPUT_CSV_DIRS[buffer_type], f"{quadkey}_land_cover_counts.csv")
    if not OVERWRITE and os.path.exists(output_csv_path):
        logging.info(f"CSV file {output_csv_path} already exists. Skipping processing.")
        return None

    # Read the polygon feather file
    polygon_path = os.path.join(POLYGON_DIRS[buffer_type], feather_file)
    try:
        polygon_df = feather.read_feather(polygon_path)
    except Exception as e:
        logging.error(f"Error reading feather file {polygon_path}: {e}")
        return None

    # Determine the correct geometry column name
    geometry_column = 'geometry_10m' if buffer_type == "10m" else 'geometry_30m'

    # Convert WKB to geometries
    try:
        polygon_df[geometry_column] = polygon_df[geometry_column].apply(wkb.loads)
    except Exception as e:
        logging.error(f"Error converting WKB to geometries for file {feather_file}: {e}")
        return None

    # Create GeoDataFrame
    gdf = gpd.GeoDataFrame(polygon_df, geometry=geometry_column)

    # Extract the original CRS from the GeoDataFrame
    original_crs = gdf.crs
    if original_crs is None:
        # Set default CRS if none is provided
        original_crs = "EPSG:4326"
        gdf.set_crs(original_crs, inplace=True)
        logging.info(f"No CRS found for GeoDataFrame. Setting CRS to {original_crs}.")

    # Get the associated raster file
    raster_path = os.path.join(TILE_DIR, f"{quadkey}.tif")
    
    if not os.path.exists(raster_path):
        logging.error(f"Error: Raster file {raster_path} not found.")
        return None

    # Print the CRS of the raster file
    try:
        with rasterio.open(raster_path) as src:
            raster_crs = src.crs
            print(f"Raster CRS: {raster_crs}")
    except Exception as e:
        logging.error(f"Error opening raster file {raster_path}: {e}")
        return None

    # Log the number of polygons
    num_polygons = len(gdf)
    logging.info(f"Number of polygons: {num_polygons}")

    # Calculate the number of splits needed
    num_splits = max(1, (num_polygons + SPLIT_SIZE - 1) // SPLIT_SIZE)

    # Split and reproject the GeoDataFrame in parts
    gdf_parts = np.array_split(gdf, num_splits)
    reprojected_parts = []
    for i, part in enumerate(gdf_parts):
        logging.info(f"Reprojecting GeoDataFrame part {i + 1} of {num_splits} from {original_crs} to {raster_crs}")
        reprojected_part = reproject_gdf(part, raster_crs)
        reprojected_parts.append(reprojected_part)

    # Debug: Print the size of each reprojected part before processing
    for i, part in enumerate(reprojected_parts):
        logging.info(f"Size of reprojected part {i + 1}: {len(part)} polygons")

    # Process each reprojected part separately
    all_results = []

    with Pool(processes=NUM_WORKERS) as pool:
        futures = []
        progress_bar = tqdm(total=len(reprojected_parts), desc="Processing splits", position=0)
        for i, part in enumerate(reprojected_parts):
            logging.info(f"Starting parallel processing of reprojected part {i + 1} of {num_splits}")
            futures.append(pool.apply_async(process_part, (part, raster_path, geometry_column)))
        
        for future in futures:
            try:
                results_df = future.get()
                all_results.append(results_df)
            except Exception as e:
                logging.error(f"Error processing part: {e}")
        progress_bar.close()

    # Filter out None results
    all_results = [result for result in all_results if result is not None]

    if not all_results:
        logging.error("No valid results to concatenate.")
        return None

    # Concatenate all results DataFrames
    final_results_df = pd.concat(all_results, ignore_index=True)

    # Rename columns to include class name and number
    final_results_df.columns = ['unique_id'] + [f"{CLASSIFICATION_NAMES[col]}-{col}" for col in final_results_df.columns[1:]]

    # Print statement before saving DataFrame to CSV
    logging.info(f"Saving DataFrame to CSV at {output_csv_path}")

    # Save DataFrame to CSV
    final_results_df.to_csv(output_csv_path, index=False)
    logging.info(f"Land cover counts saved to: {output_csv_path}")

    return final_results_df

def reproject_gdf(gdf, raster_crs):
    if gdf.crs != raster_crs:
        logging.info(f"Reprojecting GeoDataFrame from {gdf.crs} to {raster_crs}")
        gdf = gdf.to_crs(raster_crs)
    return gdf

def process_part(part, raster_path, geometry_column):
    results = {row['unique_id']: {key: 0 for key in CLASSIFICATION_NAMES.keys()} for _, row in part.iterrows()}
    for _, row in part.iterrows():
        try:
            result = process_row(row, raster_path, geometry_column)
            if result:
                unique_id, counts_dict = result
                for classification, count in counts_dict.items():
                    if classification in results[unique_id]:
                        results[unique_id][classification] += count
        except Exception as e:
            logging.error(f"Error processing polygon: {e}")
    results_df = pd.DataFrame.from_dict(results, orient='index').reset_index()
    results_df.rename(columns={'index': 'unique_id'}, inplace=True)
    return results_df

# -----------------------------
# Main Processing Loop
# -----------------------------

if __name__ == "__main__":
    # Process each feather file sequentially for both 10m and 30m buffers
    for buffer_type in ["10m", "30m"]:
        try:
            feather_files = [f for f in os.listdir(POLYGON_DIRS[buffer_type]) if f.endswith('.feather')]
        except Exception as e:
            logging.error(f"Error listing files in {POLYGON_DIRS[buffer_type]}: {e}")
            continue

        total_files = len(feather_files)
        
        if total_files == 0:
            logging.warning(f"No feather files found in {POLYGON_DIRS[buffer_type]}. Skipping this buffer type.")
            continue
        
        # Percentage of files to be plotted for quality control
        num_files_to_plot = int(total_files * PLOT_PERCENT / 100)
        plot_files = set(random.sample(feather_files, num_files_to_plot)) if PLOT_PERCENT > 0 else set()
        
        for idx, feather_file in enumerate(feather_files):
            # Optional: Implement plotting for selected files
            if feather_file in plot_files:
                polygon_path = os.path.join(POLYGON_DIRS[buffer_type], feather_file)
                try:
                    polygon_df = feather.read_feather(polygon_path)
                    geometry_column = 'geometry_10m' if buffer_type == "10m" else 'geometry_30m'
                    polygon_df[geometry_column] = polygon_df[geometry_column].apply(wkb.loads)
                    gdf = gpd.GeoDataFrame(polygon_df, geometry=geometry_column)
                    raster_path = os.path.join(TILE_DIR, f"{os.path.splitext(feather_file)[0]}.tif")
                    if os.path.exists(raster_path):
                        plot_raster_and_polygons(raster_path, gdf)
                    else:
                        logging.warning(f"Raster file {raster_path} does not exist. Skipping plot.")
                except Exception as e:
                    logging.error(f"Error during plotting for file {feather_file}: {e}")
            
            # Process the polygon file
            process_polygon_file(feather_file, idx, total_files, buffer_type)


In [ ]:
### Testing Visualizations

In [ ]:
import os
import geopandas as gpd
import pyarrow.feather as feather
import folium
from shapely import wkb
import mercantile
import rasterio
import numpy as np
from matplotlib.colors import ListedColormap, BoundaryNorm
from shapely.geometry import box
from rasterio.mask import mask
from rasterio.warp import calculate_default_transform, reproject, Resampling
import imageio
import branca
import pandas as pd
from concurrent.futures import ProcessPoolExecutor
import tempfile

# Set input and output directories
tile_dir = "/home/cbroderick/exploration/defensible-space/land_cover_tiles"
polygon_dir_30m = "/home/cbroderick/exploration/defensible-space/building_defensible_space_polygons/zoom_7_buffer_30m"
polygon_dir_original = "/home/cbroderick/exploration/defensible-space/building_defensible_space_polygons/zoom_7_original"
polygon_dir_10m = "/home/cbroderick/exploration/defensible-space/building_defensible_space_polygons/zoom_7_buffer_10m"
csv_dir = "/home/cbroderick/exploration/defensible-space/output-csvs/30m"
output_dir = "/home/cbroderick/exploration/defensible-space/output_maps"

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Define bounding boxes and names
locations = {
    "Santa Barbara": [-120, 34.455, -119.6, 34.48],
    "Sedona, Arizona": [-111.8, 34.85, -111.7, 34.9],
    "Durango, Colorado": [-107.95, 37.25, -107.85, 37.30],  # Rural area near Durango
    "Whittier, Alaska": [-148.7, 60.75, -148.65, 60.80]  # Near Whittier, Alaska
}

# Choose location
location_name = "Durango, Colorado"  # Change to "Sedona, Arizona", "Durango, Colorado", or "Anchorage, Alaska" to switch location

bounding_box = locations[location_name]
print(f"Selected location: {location_name}")
print(f"Bounding box: {bounding_box}")

# Find the mercator tile associated with the center of this bounding box
center_lon = (bounding_box[0] + bounding_box[2]) / 2
center_lat = (bounding_box[1] + bounding_box[3]) / 2
tile = mercantile.tile(center_lon, center_lat, zoom=7)

# Get the quadkey for this tile
quadkey = mercantile.quadkey(tile)
print(f"Quadkey for the tile containing the center: {quadkey}")

# Function to read and process polygon data
def read_and_process_polygons(args):
    polygon_dir, bbox, geom_col, quadkey = args
    polygon_file = os.path.join(polygon_dir, f"{quadkey}.feather")
    print(f"Looking for polygon file: {polygon_file}")

    if not os.path.exists(polygon_file):
        print(f"Error: Polygon file {polygon_file} not found.")
        return None

    print("Polygon file found. Reading data...")

    # Read the polygon feather file
    polygon_df = feather.read_feather(polygon_file)
    print(f"Number of polygons read: {len(polygon_df)}")

    # Convert WKB to geometries
    polygon_df[geom_col] = polygon_df[geom_col].apply(wkb.loads)

    # Create GeoDataFrame
    gdf = gpd.GeoDataFrame(polygon_df, geometry=geom_col, crs="EPSG:4326")

    # Subset polygons by bounding box
    subset_gdf = gdf.cx[bbox[0]:bbox[2], bbox[1]:bbox[3]]
    print(f"Number of polygons in subset: {len(subset_gdf)}")

    return subset_gdf

# Process polygons from all three datasets in parallel
with ProcessPoolExecutor(max_workers=4) as executor:
    futures = [
        executor.submit(read_and_process_polygons, (polygon_dir_30m, bounding_box, 'geometry_30m', quadkey)),
        executor.submit(read_and_process_polygons, (polygon_dir_original, bounding_box, 'geometry', quadkey)),
        executor.submit(read_and_process_polygons, (polygon_dir_10m, bounding_box, 'geometry_10m', quadkey))
    ]
    subset_gdf_30m, subset_gdf_original, subset_gdf_10m = [future.result() for future in futures]

print("Reading raster data...")

# Create a temporary directory
with tempfile.TemporaryDirectory() as temp_dir:
    print(f"Temporary directory created at: {temp_dir}")

    # Read the raster file and check its CRS
    raster_file_name = f"{quadkey}.tif"
    raster_path = os.path.join(tile_dir, raster_file_name)
    if not os.path.exists(raster_path):
        raise FileNotFoundError(f"Raster file {raster_path} not found.")

    with rasterio.open(raster_path) as src:
        original_raster_data = src.read(1)
        raster_crs = src.crs
        print(f"Raster CRS: {raster_crs}")

        # Print out the total count of each land cover class within the original raster
        unique, counts = np.unique(original_raster_data, return_counts=True)
        land_cover_counts = dict(zip(unique, counts))
        
        classification_names = {0: 'No Data 1', 1: 'Water', 2: 'Trees', 4: 'Flooded vegetation', 5: 'Crops',
                                7: 'Built area', 8: 'Bare ground', 9: 'Snow/ice', 10: 'Clouds', 11: 'Rangeland',
                                255: 'No Data 2'}
        print("Land cover counts in the original raster:")
        for k, v in land_cover_counts.items():
            class_name = classification_names.get(k, f"Class {k}")
            print(f"{class_name}: {v} pixels")

        # Ensure the raster CRS is known
        if raster_crs is None:
            raise ValueError("The raster does not have a CRS.")

    # Reproject the entire raster to WGS84 (EPSG:4326)
    dst_crs = 'EPSG:4326'
    reprojected_raster_path = os.path.join(temp_dir, 'reprojected_raster.tif')
    with rasterio.open(raster_path) as src:
        transform, width, height = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds)
        kwargs = src.meta.copy()
        kwargs.update({
            'crs': dst_crs,
            'transform': transform,
            'width': width,
            'height': height,
            'dtype': 'int32',
            'nodata': 255  # Specify No Data value
        })

        reprojected_raster_data = np.empty((height, width), dtype=np.int32)
        with rasterio.open(reprojected_raster_path, 'w', **kwargs) as dst:
            for i in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, i),
                    destination=reprojected_raster_data,
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=dst_crs,
                    resampling=Resampling.nearest,
                    src_nodata=255,  # Handle No Data values
                    dst_nodata=255)
                dst.write(reprojected_raster_data, i)

    # Crop the reprojected raster using the WGS84 bounding box
    geom = gpd.GeoDataFrame({'geometry': [box(*bounding_box)]}, crs='EPSG:4326')

    with rasterio.open(reprojected_raster_path) as src:
        out_image, out_transform = mask(src, geom.geometry, crop=True, nodata=255)
        out_meta = src.meta.copy()

    out_meta.update({
        "driver": "GTiff",
        "height": out_image.shape[1],
        "width": out_image.shape[2],
        "transform": out_transform,
        'dtype': 'int32',
        'nodata': 255
    })

    # Save the cropped raster to a new file
    cropped_raster_path = os.path.join(temp_dir, 'cropped_raster.tif')
    with rasterio.open(cropped_raster_path, "w", **out_meta) as dest:
        dest.write(out_image)

    # Convert the cropped raster to a color-mapped image
    with rasterio.open(cropped_raster_path) as src:
        cropped_image = src.read(1)

    # Define the color key and classification names
    color_key = {
        0: '#000000',  # No Data 1
        1: '#0000FF',  # Water
        2: '#00FF00',  # Trees
        4: '#800080',  # Flooded vegetation
        5: '#FFFF00',  # Crops
        7: '#808080',  # Built area
        8: '#8B4513',  # Bare ground
        9: '#FFFFFF',  # Snow/ice
        10: '#D3D3D3',  # Clouds
        11: '#D2B48C',  # Rangeland
        255: '#FF0000'  # No Data 2 (different color from 0)
    }

    # Define the discrete colormap and normalization
    cmap = ListedColormap([color_key[key] for key in sorted(color_key.keys())])
    norm = BoundaryNorm(sorted(color_key.keys()) + [max(color_key.keys()) + 1], cmap.N)

    # Apply color mapping
    color_mapped_image = cmap(norm(cropped_image))

    # Save the color-mapped image as a PNG file
    png_path = os.path.join(temp_dir, 'cropped_raster.png')
    imageio.imwrite(png_path, (color_mapped_image[:, :, :3] * 255).astype(np.uint8))

    print("Creating map...")

    # Read the CSV file and merge it with the original polygons
    csv_file_path = os.path.join(csv_dir, f"{quadkey}_land_cover_counts.csv")
    if os.path.exists(csv_file_path):
        csv_df = pd.read_csv(csv_file_path)
        merged_gdf_original = subset_gdf_original.merge(csv_df, on='unique_id', how='left')
    else:
        print(f"Warning: CSV file {csv_file_path} not found. Proceeding without land cover counts.")
        merged_gdf_original = subset_gdf_original

    # Create a folium map centered around the selected location
    m = folium.Map(location=[center_lat, center_lon], zoom_start=12)

    # Add the raster image to the map using the calculated bounds
    with rasterio.open(cropped_raster_path) as src:
        bounds = src.bounds
        bottom, left, top, right = bounds.bottom, bounds.left, bounds.top, bounds.right

    folium.raster_layers.ImageOverlay(
        name='Land Cover',
        image=png_path,
        bounds=[[bottom, left], [top, right]],
        opacity=0.8,
        interactive=True,
        cross_origin=False,
        zindex=1
    ).add_to(m)

    # Function to create tooltip content
    def create_tooltip(row):
        tooltip_html = "<b>Polygon Data:</b><br>"
        for col in row.index:
            if col not in ['geometry', 'unique_id']:
                tooltip_html += f"{col}: {row[col]}<br>"
        return tooltip_html

    # Add other polygon layers to the map first
    folium.GeoJson(
        subset_gdf_30m,
        name='Defensible Space Polygons (30m buffer)',
        style_function=lambda x: {'fillColor': 'red', 'color': 'black', 'weight': 1, 'fillOpacity': 0.2}
    ).add_to(m)

    folium.GeoJson(
        subset_gdf_10m,
        name='Defensible Space Polygons (10m buffer)',
        style_function=lambda x: {'fillColor': 'blue', 'color': 'black', 'weight': 1, 'fillOpacity': 0.2}
    ).add_to(m)

    # Add original polygons to the map with tooltips last
    for _, row in merged_gdf_original.iterrows():
        folium.GeoJson(
            row['geometry'],
            name='Defensible Space Polygons (Original)',
            style_function=lambda x: {'fillColor': 'black', 'color': 'black', 'weight': 1, 'fillOpacity': 0.5},
            tooltip=folium.Tooltip(create_tooltip(row)),
            popup=folium.Popup(create_tooltip(row), max_width=300)
        ).add_to(m)

    # Create a categorical legend
    legend_html = '''
    <div style="position: fixed;
         bottom: 50px; left: 50px; width: 200px; height: 280px;
         border:2px solid grey; z-index:9999; font-size:14px;
         background-color:white; opacity: 0.9;
         padding: 10px;">
         <b>Land Cover Legend</b><br>
         <i style="background: #000000; width: 18px; height: 18px; float: left; margin-right: 8px; opacity: 0.7;"></i>No Data 1<br>
         <i style="background: #0000FF; width: 18px; height: 18px; float: left; margin-right: 8px; opacity: 0.7;"></i>Water<br>
         <i style="background: #00FF00; width: 18px; height: 18px; float: left; margin-right: 8px; opacity: 0.7;"></i>Trees<br>
         <i style="background: #800080; width: 18px; height: 18px; float: left; margin-right: 8px; opacity: 0.7;"></i>Flooded vegetation<br>
         <i style="background: #FFFF00; width: 18px; height: 18px; float: left; margin-right: 8px; opacity: 0.7;"></i>Crops<br>
         <i style="background: #808080; width: 18px; height: 18px; float: left; margin-right: 8px; opacity: 0.7;"></i>Built area<br>
         <i style="background: #8B4513; width: 18px; height: 18px; float: left; margin-right: 8px; opacity: 0.7;"></i>Bare ground<br>
         <i style="background: #FFFFFF; width: 18px; height: 18px; float: left; margin-right: 8px; opacity: 0.7;"></i>Snow/ice<br>
         <i style="background: #D3D3D3; width: 18px; height: 18px; float: left; margin-right: 8px; opacity: 0.7;"></i>Clouds<br>
         <i style="background: #D2B48C; width: 18px; height: 18px; float: left; margin-right: 8px; opacity: 0.7;<i style="background: #D2B48C; width: 18px; height: 18px; float: left; margin-right: 8px; opacity: 0.7;"></i>Rangeland<br>
         <i style="background: #FF0000; width: 18px; height: 18px; float: left; margin-right: 8px; opacity: 0.7;"></i>No Data 2<br>
    </div>
    '''

    # Add the legend to the map
    m.get_root().html.add_child(folium.Element(legend_html))

    # Add a layer control panel
    folium.LayerControl().add_to(m)

    # Save the map to an HTML file with the name of the location
    html_path = os.path.join(output_dir, f"{location_name.replace(', ', '_').replace(' ', '_').lower()}_combined_map.html")
    m.save(html_path)

    print(f"Combined map saved as: {html_path}")


In [ ]:
import os

# Define the old and new folder paths
old_folder = os.path.join(WRI_PROJECT_ROOT, 'data', 'built-environment-domain-data', 'defensible-space', 'structure-landcover-classes-csvs')
new_folder = os.path.join(WRI_PROJECT_ROOT, 'data', 'built-environment-domain-data', 'defensible-space', 'structure-landcover-counts')

# Rename the folder
try:
    os.rename(old_folder, new_folder)
    print(f"Successfully renamed folder to: {new_folder}")
except FileNotFoundError:
    print("Error: The original folder does not exist.")
except PermissionError:
    print("Error: Permission denied. Try running with appropriate permissions.")
except Exception as e:
    print(f"An error occurred: {e}")
